# Environnement ai-toolkit (image officielle)

Pod sur l'image `ostris/aitoolkit:latest` : ai-toolkit est deja installe, l'interface demarre toute seule sur le port 8675. JupyterLab tourne en parallele sur le port 8888. Rien a installer.

**Routine**
1. Cellule **1** : etat du pod et lien de l'interface.
2. Cellule **2** : pre-telecharger Krea 2 pendant que tu prepares le reste (optionnel).
3. Cellule **3** : recuperer tes datasets depuis HF. Cellule **4** si tu dois captionner sur place.
4. Entrainer depuis l'interface (chemins `/app/ai-toolkit/...`, comme sur le template public).
5. Cellule **5** : envoyer les checkpoints sur HF, puis Terminate.

## 1. Etat du pod

In [ ]:
%%bash
echo "GPU      : $(nvidia-smi --query-gpu=name,driver_version --format=csv,noheader)"
python3 -c "import torch;print('torch    :',torch.__version__,'| CUDA OK' if torch.cuda.is_available() else '| CUDA KO')"
echo "ai-toolkit : $(git -C /app/ai-toolkit log -1 --format='%h du %cd' --date=short 2>/dev/null)"
if [ -n "$HF_TOKEN" ]; then echo "HF       : $(hf auth whoami 2>/dev/null | head -1)"; else echo "HF       : !! HF_TOKEN absent du template"; fi
echo "Cache HF : ${HF_HOME:-/root/.cache/huggingface}"
df -h /workspace | tail -1 | awk '{print "Volume   : " $4 " libres sur " $2}'
df -h / | tail -1 | awk '{print "Container: " $4 " libres sur " $2}'
if curl -s -o /dev/null --max-time 3 http://127.0.0.1:8675; then echo "Interface: OK"; else echo "Interface: pas encore prete (relance dans 30 s)"; fi
echo ""
echo "LIEN : https://${RUNPOD_POD_ID}-8675.proxy.runpod.net"

## 2. Pre-telecharger les modeles (optionnel)
En arriere-plan. ai-toolkit retrouvera les fichiers dans le cache au lancement du job. Relance la cellule pour voir l'avancement.

In [ ]:
%%bash
# ======================= REGLAGES =======================
#   krea2     Krea 2 Raw + encodeur Qwen3-VL-4B + VAE Qwen-Image   ~36 Go
#   caption   Qwen3-VL-8B pour le captioning (cellule 4)           ~18 Go
export PREFETCH="krea2"
# ========================================================
export HF_XET_HIGH_PERFORMANCE=1
LOG=/workspace/prefetch.log
if [ -f "$LOG" ] && ! grep -q PREFETCH_TERMINE "$LOG" && pgrep -f "hf download" >/dev/null; then
  echo "En cours : $(grep -- '^---' "$LOG" | tail -1 | cut -c5-)"
elif [ -f "$LOG" ] && grep -q PREFETCH_TERMINE "$LOG"; then
  echo "Termine."
else
  setsid nohup bash -c '
    if [[ ",$PREFETCH," == *",krea2,"* ]]; then
      echo "--- Krea 2 Raw";     hf download krea/Krea-2-Raw raw.safetensors
      echo "--- Qwen3-VL-4B";    hf download Qwen/Qwen3-VL-4B-Instruct
      echo "--- VAE Qwen-Image"; hf download Qwen/Qwen-Image --include "vae/*"
    fi
    if [[ ",$PREFETCH," == *",caption,"* ]]; then
      echo "--- Qwen3-VL-8B";    hf download Qwen/Qwen3-VL-8B-Instruct
    fi
    echo PREFETCH_TERMINE
  ' > "$LOG" 2>&1 &
  echo "Lance ($PREFETCH)."
fi
grep -iE "error|denied|403|gated" "$LOG" 2>/dev/null | tail -3
du -sh "${HF_HOME:-/root/.cache/huggingface}" 2>/dev/null | awk '{print "Cache HF : " $1}'

## 3. Recuperer des datasets depuis Hugging Face
Depuis `rafatribe/Personnal_Loras`, dossier `datasets/<nom>`. Un dataset deja present n'est pas ecrase.

In [ ]:
%%bash
# ======================= REGLAGES =======================
export DATASETS="clementine_paysages,clementine_recadrages"
# ========================================================
export HF_XET_HIGH_PERFORMANCE=1
DEST=/app/ai-toolkit/datasets
TMP=/workspace/hf_tmp
mkdir -p "$DEST"
for d in ${DATASETS//,/ }; do
  if [ -d "$DEST/$d" ]; then echo "-- $d : deja present, garde tel quel"; continue; fi
  echo "Telechargement de $d..."
  hf download rafatribe/Personnal_Loras --include "datasets/$d/*" --local-dir "$TMP" >/dev/null 2>&1
  if [ -d "$TMP/datasets/$d" ]; then
    mv "$TMP/datasets/$d" "$DEST/"
    n_img=$(ls "$DEST/$d" | grep -ciE '\.(png|jpe?g|webp)$')
    n_txt=$(ls "$DEST/$d" | grep -ci '\.txt$')
    echo "OK $d : $n_img images, $n_txt captions"
  else
    echo "!! $d : introuvable sur HF (verifie le nom)"
  fi
done
rm -rf "$TMP"
echo ""
echo "Datasets sur le pod :"; ls "$DEST"

## 4. Captionner un dataset sur place (optionnel)
Qwen3-VL-8B, captions au format ai-toolkit (`TRIGGER, description du contenu`, sans jamais nommer le style). Ne traite que les images sans `.txt`. Aucun job ne doit tourner pendant le captioning (VRAM).

In [ ]:
%%bash
# ======================= REGLAGES =======================
export DOSSIER="/app/ai-toolkit/datasets/mon_dataset"
export TRIGGER="mon_style"
export ENVOYER_SUR_HF="oui"   # oui = sauvegarde le dataset captionne dans datasets/<nom> sur HF
# ========================================================
export HF_XET_HIGH_PERFORMANCE=1
[ -d "$DOSSIER" ] || { echo "Dossier introuvable : $DOSSIER"; ls /app/ai-toolkit/datasets; exit 1; }
USED=$(nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits | head -1)
[ "$USED" -gt 2000 ] && { echo "!! ${USED} Mo de VRAM deja utilises : un job tourne ? Arrete-le avant."; exit 1; }

cat > /workspace/caption.py <<'PY'
import os, re
from pathlib import Path
import torch
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

DATASET = Path(os.environ["DOSSIER"]); TRIGGER = os.environ["TRIGGER"]
MODEL = "Qwen/Qwen3-VL-8B-Instruct"
PROMPT = (
    "Describe this image as a caption for training an image generation model. "
    "Write one paragraph of 2 to 4 plain sentences. Describe only the content: "
    "the subjects (species, age, hair, clothing and their colors), their pose, action and expression, "
    "the setting and its objects, the framing (close-up, medium shot, wide shot), the camera angle, "
    "and the lighting or time of day. "
    "Do NOT mention the art style, the medium or the era: never use words like cartoon, animation, animated, "
    "cel, drawing, drawn, illustration, illustrated, painting, painted, vintage, retro, 1980s, frame, still, screenshot. "
    "Do not name characters or shows. Do not start with 'The image shows' or 'This image'. "
    "Start directly with the main subject."
)
BANNED = re.compile(r"\b(cartoon\w*|animat\w*|cel\b|drawing|drawn|illustrat\w*|paint\w*|vintage|retro|1980s|80s|screenshot|still frame)\b", re.I)

def clean(t):
    t = re.sub(r"\s+", " ", t.strip())
    t = re.sub(r"^(the image (shows|depicts)|this image (shows|depicts)|in this image,?)\s*", "", t, flags=re.I)
    return t[:1].upper() + t[1:] if t else t

imgs = sorted(p for p in DATASET.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"})
todo = [p for p in imgs if not p.with_suffix(".txt").exists()]
print(f"{len(imgs)} images, {len(todo)} a captionner.")
if todo:
    model = Qwen3VLForConditionalGeneration.from_pretrained(MODEL, dtype=torch.bfloat16, device_map="cuda")
    proc = AutoProcessor.from_pretrained(MODEL)
    flagged = []
    for i, img in enumerate(todo, 1):
        msgs = [{"role": "user", "content": [{"type": "image", "image": str(img)}, {"type": "text", "text": PROMPT}]}]
        inp = proc.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=220, do_sample=False)
        desc = clean(proc.batch_decode(out[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0])
        img.with_suffix(".txt").write_text(f"{TRIGGER}, {desc}", encoding="utf-8")
        if BANNED.search(desc): flagged.append(img.name)
        print(f"[{i}/{len(todo)}] {img.name}: {desc[:100]}", flush=True)
    if flagged:
        print(f"\n{len(flagged)} captions avec un mot de style a corriger :")
        for n in flagged: print("  " + n)
print("Termine.")
PY

python3 /workspace/caption.py || exit 1
if [ "$ENVOYER_SUR_HF" = "oui" ]; then
  NOM=$(basename "$DOSSIER")
  hf upload rafatribe/Personnal_Loras "$DOSSIER" "datasets/$NOM" --commit-message "Dataset $NOM" >/dev/null 2>&1 \
    && echo "Sauvegarde sur HF : datasets/$NOM" || echo "!! Envoi sur HF en echec"
fi

## 5. Envoyer les checkpoints sur Hugging Face
Envoie les `.safetensors`, la config et les echantillons du job dans `Loras/<job>` (sans `optimizer.pt`).

In [ ]:
%%bash
# ======================= REGLAGES =======================
export JOB="clementine_style_krea2_v2"
# ========================================================
export HF_XET_HIGH_PERFORMANCE=1
OUT=/app/ai-toolkit/output/$JOB
[ -d "$OUT" ] || { echo "Job introuvable. Jobs presents :"; ls /app/ai-toolkit/output; exit 1; }
echo "$(ls "$OUT"/*.safetensors 2>/dev/null | wc -l) checkpoints a envoyer..."
hf upload rafatribe/Personnal_Loras "$OUT" "Loras/$JOB" \
  --include "*.safetensors" --include "*.yaml" --include "samples/*" \
  --commit-message "LoRA $JOB" >/dev/null 2>&1 \
  && echo "OK : https://huggingface.co/rafatribe/Personnal_Loras/tree/main/Loras/$JOB" \
  || echo "!! Envoi en echec"
echo ""
echo "Sur le pod ComfyUI :"
echo "  hf download rafatribe/Personnal_Loras --include \"Loras/$JOB/*.safetensors\" --local-dir <dossier loras>"